In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns 
import string
import torch
import re
import random, os
import emoji

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from transformers import AutoTokenizer, BertModel, BertTokenizer, AutoModelForSeq2SeqLM
from torch.utils.data import Dataset
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from transformers import AutoModelForSequenceClassification, AutoConfig, BertForSequenceClassification


import torch.nn as nn
from transformers import EarlyStoppingCallback
import transformers
from transformers import Trainer, TrainingArguments
from datasets import Dataset as HFDataset
from sklearn.metrics import classification_report
import os, re, random
import numpy as np
import pandas as pd
import torch
import emoji
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

import nltk
import nlpaug.augmenter.sentence as nas
import nlpaug.augmenter.word as naw
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from nlpaug.augmenter.word import BackTranslationAug

In [18]:
tqdm.pandas()
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

Device: cpu


In [20]:
df = pd.read_csv(r"C:\Users\Aditya P J\Documents\Kuliah\Semester 7\Forecast\Proyek\Data\scraping_cnbc_BBCA_sorted.csv")
df.head()

,Title,Date,DayOfWeek
0,"Kena Lempar Botol, Satu Pengunjung Jakarta Nig...",2015-01-01,Thursday
1,"Harga BBM Sudah Turun, Tapi Tarif Angkutan Sul...",2015-01-04,Sunday
2,Pencabutan Subsidi Premium Bisa Bikin Dolar Ke...,2015-01-04,Sunday
3,"Subsidi Premium Dicabut, Ekonom BCA: Market Me...",2015-01-04,Sunday
4,Ini Alasan Mengapa Anak Sebaiknya Tak Diizinka...,2015-01-06,Tuesday


In [22]:
df['Date'] = pd.to_datetime(df['Date'])

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11059 entries, 0 to 11058
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Title      11035 non-null  object        
 1   Date       11059 non-null  datetime64[ns]
 2   DayOfWeek  11059 non-null  object        
dtypes: datetime64[ns](1), object(2)
memory usage: 259.3+ KB


In [26]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

In [28]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    s = text

    # 1. Lowercase
    s = s.lower()
    # 2. Hapus URL, email, mention, hashtag (atau kamu bisa keep hashtag jika penting)
    s = re.sub(r'http\S+|www\.\S+', ' ', s)
    s = re.sub(r'\S+@\S+', ' ', s)
    s = re.sub(r'@\w+', ' ', s)
    s = re.sub(r'#\w+', ' ', s)
    # 3. Hapus HTML tags
    s = re.sub(r'<.*?>', ' ', s)
    # 4. Ubah repeated punctuation/char (mis. "sooooo" -> "soo" atau biarkan satu)
    s = re.sub(r'(.)\1{2,}', r'\1\1', s)
    # 5. Hapus extra whitespace & strip
    s = re.sub(r'\s+', ' ', s).strip()
    # 6. convert emoji → teks
    s = emoji.demojize(s, delimiters=(" ", " "))  

    return s

In [30]:
df['Title'] = df['Title'].apply(clean_text)
df.head()

,Title,Date,DayOfWeek
0,"kena lempar botol, satu pengunjung jakarta nig...",2015-01-01,Thursday
1,"harga bbm sudah turun, tapi tarif angkutan sul...",2015-01-04,Sunday
2,pencabutan subsidi premium bisa bikin dolar ke...,2015-01-04,Sunday
3,"subsidi premium dicabut, ekonom bca: market me...",2015-01-04,Sunday
4,ini alasan mengapa anak sebaiknya tak diizinka...,2015-01-06,Tuesday


In [31]:
def load_kamus(file_path):
    kamus={}
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                slang = parts[0]
                normal = " ".join(parts[1:])
                kamus[slang] = normal
    return kamus

In [32]:
def normalisasi_teks(teks, kamus):
    words = teks.split()
    return " ".join([kamus.get(w, w) for w in words])

In [33]:
kamus = load_kamus(r"C:\Users\Aditya P J\Documents\Kuliah\Skripsi\Data\kbba.txt")
df['Title'] = df['Title'].apply(lambda x: normalisasi_teks(x, kamus))
df.head(20)

,Title,Date,DayOfWeek
0,"kena lempar botol, satu pengunjung jakarta nig...",2015-01-01,Thursday
1,"harga bbm sudah turun, tapi tarif angkutan sul...",2015-01-04,Sunday
2,pencabutan subsidi premium bisa bikin dolar ke...,2015-01-04,Sunday
3,"subsidi premium dicabut, ekonom bca: market me...",2015-01-04,Sunday
4,ini alasan mengapa anak sebaiknya tidak diizin...,2015-01-06,Tuesday
5,waterfront securities: profit taking berpotens...,2015-01-06,Tuesday
6,waterfront securities: ihsg cenderung mix,2015-01-07,Wednesday
7,"sudah bayar, oknum kolektor hsbc menagih tanpa...",2015-01-07,Wednesday
8,"ayo bantu, anak-anak tpa di cileungsi bogor in...",2015-01-07,Wednesday
9,proses kta bca yang tidak kunjung selesai,2015-01-08,Thursday


In [34]:
model_name = "w11wo/indonesian-roberta-base-sentiment-classifier"
tokenizer = AutoTokenizer.from_pretrained(model_name)
clf_model = AutoModelForSequenceClassification.from_pretrained(model_name)
clf_model.to(device)
clf_model.eval()

# If you want to access encoder only (for embeddings)
# On many HF RoBERTa-based models the encoder is clf_model.roberta or clf_model.base_model
# We'll try both safely:
if hasattr(clf_model, "roberta"):
    encoder = clf_model.roberta
elif hasattr(clf_model, "base_model"):
    encoder = clf_model.base_model
else:
    encoder = None  # fallback (rare)

if encoder is not None:
    encoder.to(device)
    encoder.eval()


D:\Terminal\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [39]:
def get_embeddings(text, pooling="cls"):
    """
    pooling: "cls" or "mean"
    returns numpy array (1D) of size hidden_size
    """
    # tokenizer params: fixed-length for stability
    inputs = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=512,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k,v in inputs.items()}

    # Prefer encoder if present (so we don't run classifier head)
    with torch.no_grad():
        if encoder is not None:
            out = encoder(**inputs, output_hidden_states=True, return_dict=True)
            # last hidden
            last_hidden = out.hidden_states[-1]  # [1, seq_len, hidden_size]
        else:
            # fallback: run full model and take last_hidden_state if available
            full_out = clf_model.base_model(**inputs, output_hidden_states=True, return_dict=True)
            last_hidden = full_out.hidden_states[-1]

    if pooling == "cls":
        emb = last_hidden[:, 0, :]  # [1, hidden]
    else:  # mean pooling over non-padded tokens
        attention_mask = inputs.get("attention_mask")
        if attention_mask is None:
            emb = last_hidden.mean(dim=1)
        else:
            mask = attention_mask.unsqueeze(-1)  # [1, seq_len, 1]
            masked = last_hidden * mask
            summed = masked.sum(dim=1)  # [1, hidden]
            denom = mask.sum(dim=1).clamp(min=1e-9)
            emb = summed / denom

    return emb.squeeze(0).cpu().numpy()  # return 1D numpy


In [46]:
def predict_label(text):
    inputs = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=512,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k,v in inputs.items()}
    with torch.no_grad():
        logits = clf_model(**inputs).logits  # [1, num_labels]
    pred_id = int(torch.argmax(logits, dim=1).cpu().item())
    # Resolve label name robustly
    id2label = clf_model.config.id2label  # dict
    label = id2label.get(pred_id, str(pred_id))
    return label


In [50]:
def extract_token_and_embedding_row(text, pooling="cls"):
    # tokens
    inputs = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=512,
        return_tensors="pt"
    )
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze(0))
    emb = get_embeddings(text, pooling=pooling)
    return {"tokens": tokens, "embedding": emb}

# apply (this may take time)
df["Roberta_Result"] = df["Title"].progress_apply(lambda x: extract_token_and_embedding_row(x, pooling="cls"))
df["Tokens"] = df["Roberta_Result"].apply(lambda x: x["tokens"])
df["Embedding"] = df["Roberta_Result"].apply(lambda x: x["embedding"])

# sentiment prediction (tokenizer -> classifier)
df["SentimentLabel"] = df["Title"].progress_apply(predict_label)


  5%|███▉                                                                        | 577/11059 [09:01<2:43:52,  1.07it/s]

KeyboardInterrupt



In [ ]:
# ----------------------------
def label_to_score(label):
    # lower and check substrings
    l = str(label).lower()
    if "pos" in l or "positive" in l:
        return 1
    if "neg" in l or "negative" in l:
        return -1
    if "neu" in l or "neutral" in l or "netral" in l:
        return 0
    # fallback: if label is numeric string
    try:
        if int(l) > 0:
            return 1
    except:
        pass
    return 0

df["SentimentScore"] = df["SentimentLabel"].apply(label_to_score)


In [ ]:
# ----------------------------
daily_sentiment = df.groupby("Date")["SentimentScore"].mean().reset_index(name="SentimentScore")
daily_count = df.groupby("Date")["SentimentScore"].count().reset_index(name="NumComments")
daily = pd.merge(daily_sentiment, daily_count, on="Date", how="outer")

# fill full date range
full_dates = pd.DataFrame({"Date": pd.date_range(df["Date"].min(), df["Date"].max())})
daily = full_dates.merge(daily, on="Date", how="left")
daily["SentimentScore"].fillna(0, inplace=True)
daily["NumComments"].fillna(0, inplace=True)
daily["HasComment"] = (daily["NumComments"] > 0).astype(int)

In [ ]:
# === 5. Agregasi harian ===
# Rata-rata skor sentimen per hari
daily_sentiment = df.groupby("Date")["SentimentScore"].mean().reset_index()

# Jumlah komentar per hari
daily_count = df.groupby("Date")["SentimentScore"].count().reset_index().rename(columns={"SentimentScore": "NumComments"})

# Gabungkan
daily = pd.merge(daily_sentiment, daily_count, on="Date")

# Lengkapi semua tanggal
full_dates = pd.DataFrame({"Date": pd.date_range(df["Date"].min(), df["Date"].max())})
daily = full_dates.merge(daily, on="Date", how="left")

# Isi missing
daily["SentimentScore"].fillna(0, inplace=True)  # hari tanpa komentar = netral
daily["NumComments"].fillna(0, inplace=True)     # hari tanpa komentar = 0
daily["HasComment"] = (daily["NumComments"] > 0).astype(int)

daily.head(15)


In [ ]:
# ----------------------------
daily.to_csv(r"C:\Users\Aditya P J\Documents\Kuliah\Semester 7\Forecast\Proyek\Data\daily_sentiment.csv", index=False)
print("Saved daily_sentiment.csv — rows:", len(daily))

# show head
print(df[["Title","SentimentLabel"]].head(10))
print(daily.head(10))

In [ ]:
plt.figure(figsize=(14,6))
plt.plot(daily["Date"], daily["SentimentScore"], label="Sentiment Score", color="blue")
plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.title("Daily Sentiment Score")
plt.xlabel("Date")
plt.ylabel("Sentiment Score (-1 Negatif, 0 Netral, 1 Positif)")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(14,6))
plt.bar(daily["Date"], daily["NumComments"], color="orange")
plt.title("Daily Number of Comments")
plt.xlabel("Date")
plt.ylabel("Number of Comments")
plt.show()


In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(daily["NumComments"], daily["SentimentScore"], alpha=0.6, c=daily["SentimentScore"], cmap="coolwarm")
plt.colorbar(label="Sentiment Score")
plt.title("Relationship Between Number of Comments and Sentiment Score")
plt.xlabel("Number of Comments")
plt.ylabel("Sentiment Score")
plt.show()
